[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tricycle/tricycle.ipynb)

# A Tricycle, in the Plane and in Space

This example drives a tricycle steered by its front wheel. A wheel that rolls without sliding moves along its own heading, so at every moment each wheel turns about a point on the line of its axle; all three wheels roll only if all three axles pass through one point, the turning centre, and the whole tricycle turns about it. Steering moves that centre, from far away when the front wheel is nearly straight to close by when it is turned hard.

Each axle is a hyperplane of projective geometric algebra, a vector: a line in the plane, a vertical plane in space. The turning centre is where the axles meet, `rear ^ front`, a point in the plane and a vertical line in space, and the tricycle moves by its exponential: a rotation about it, or a translation when the axles are parallel and their meet lies at infinity. Steering turns the front axle by a motor, `steer >> rest`, and a drive is the running product of the steps' motors. None of these lines names the dimension, and the same module drives the tricycle in the plane and in space.

In the notation of vehicle dynamics this reads as Ackermann steering: the instantaneous centre of rotation lies on the extension of every axle. The construction follows the [tricycle example of ganja.js](https://github.com/enkimute/ganja.js/blob/master/examples/example_chapter11_tricycle.html) by Steven De Keninck.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import PGA2D, PGA3D
from examples import instantiate
from examples.animation import show_animation
from examples.mechanics.tricycle import render, scenarios

np.set_printoptions(precision=4, suppress=True)

## 1. The axles meet in the turning centre

The rear axle runs across the tricycle at its origin, and the front axle at rest a wheelbase ahead, parallel to it. Steering turns the front axle about the steering axis, where the front axle meets the hyperplane down the middle of the frame. Where the two axles meet is the turning centre: its weight is the sine of the steering angle, so straight ahead it lies at infinity, and its distance from the front wheel is the wheelbase over that sine.

In [ ]:
# The distance between the axles, and a few steering angles, straight ahead to sharply left.
wheelbase = 1.2
steering = np.radians([0.0, 10.0, 30.0, 60.0])

In [ ]:
plane = instantiate("examples.mechanics.tricycle.core", PGA2D)
mv = plane.mv
rear = mv.x                                                                     # [] Plane
rest = mv.x - wheelbase * mv.w                                                  # [] Plane
# The steering axis, where the front axle at rest meets the middle of the frame, y = 0.
axis = rest ^ mv.y                                                              # [] Meet
steer = (axis.normalized() * (-steering / 2)).exp()                             # [angles] Motor
front = steer >> rest                                                           # [angles] Plane
centres = rear ^ front                                                          # [angles] Meet

In [ ]:
print("weight of each turning centre, against the sine of the steering:", (mv.w & centres).to_array(), np.sin(steering))
print("where each turning centre lies:\n", render.ground(centres))

## 2. A drive in the plane

Over a drive the steering swings to and fro about a left turn. Each step moves the tricycle in its own frame, by the exponential of the turning centre at that step's steering, and the steps multiply on the right of the pose. Below, the axles are drawn on to the turning centre, where they always meet, and the grey trail is the middle of the rear axle.

In [ ]:
plane_drive = scenarios.run(plane)                                              # contacts, points ahead, centres, rear middles
show_animation(render.animate(render.plane, scenarios.moments(*plane_drive, scenarios.N_FRAMES)),
               "tricycle_plane", scenarios.FRAME_DURATION)

## 3. The same drive in space

In the projective algebra of space the same lines give vertical planes for axles, and the turning centre is the vertical line where they meet; drawn, it stands where it crosses the ground. The tricycle leaves the same track as in the plane, to round-off.

In [ ]:
space = instantiate("examples.mechanics.tricycle.core", PGA3D)
space_drive = scenarios.run(space)                                              # contacts, points ahead, centres, rear middles
show_animation(render.animate(render.space, scenarios.moments(*space_drive, scenarios.N_FRAMES)),
               "tricycle_space", scenarios.FRAME_DURATION)

In [ ]:
print("largest difference between the tracks in the plane and in space:",
      np.abs(render.ground(plane_drive[3]) - render.ground(space_drive[3])).max())

In [ ]:
# checks
# The turning centre's weight is the sine of the steering angle, so straight ahead the axles meet at
# infinity; no wheel slides sideways, each contact, the point ahead of it and the next contact joining to
# nothing but for the step's curvature; both drives leave the same track.
np.testing.assert_allclose((mv.w & centres).to_array(), np.sin(steering), atol=1e-12)
for contacts, forward, _, _ in (plane_drive, space_drive):
    rolling = (contacts[:-1].normalized() & forward[:-1].normalized() & contacts[1:].normalized()).norm().to_array()
    assert rolling.max() < 0.1 * scenarios.WHEELBASE * (scenarios.STEP * scenarios.WHEELBASE) ** 2
np.testing.assert_allclose(render.ground(plane_drive[3]), render.ground(space_drive[3]), atol=1e-9)